# Rabi oscillations of a spin-1/2

We build a single-spin Hamiltonian, explain the rotating-wave approximation (RWA), and simulate square RF pulses with `CoherentControl.py`. Fits extract the oscillation period and amplitude on resonance and across a detuning map.

This is an **ideal coherent benchmark**: the spin starts in its ground state and `collapse_ops=[]`. Relaxation, dephasing, thermal preparation, and tunnel-current readout are intentionally excluded.

In [ ]:
# Relevant imports
import matplotlib.pyplot as plt
import numpy as np
from qutip import Qobj
from scipy.optimize import curve_fit

from spinfinity import CC, SpinSys, hamil

## 1. Building the spin Hamiltonian

For `tip="f"`, the effective static field is $\mathbf B_{\mathrm{eff}}=\mathbf B+\mathbf B_{\mathrm{Tip}}$. Spinfinity uses dimensionless spin operators and

$$
H_0=-g\mu_B\mathbf B_{\mathrm{eff}}\cdot\mathbf S.
$$

We choose $B_z=0.5$ T, $g=2$, and the small transverse tip field from the existing example. The resonance frequency is calculated from the eigenenergy splitting, using the package's own energy-to-frequency conversion.

In [ ]:
spin = SpinSys([0.5])
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T
spin.tip = "f"

hamil.calc_EigEnergies(spin, rebuildH=True)
f_res = float(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])  # GHz

# Prepare the actual ground eigenstate, not a thermal mixture.
spin.psi0 = Qobj(spin.eigVectors[:, 0])
spin.collapse_ops = []
spin.f_RF = f_res
rabi_rate = 10e-3  # GHz = 10 MHz
spin.RabiRate = rabi_rate

# Use the same quantization axis for pulses and readout.
CC.RotFrameSecular(spin, spin.f_RF)
Sz_readout = Qobj(spin.Sz_rot)

pulsewidth_array = np.linspace(0.0, 400.0, 1601)  # ns; includes zero
solver_options = {"method": "diag", "store_states": False, "progress_bar": ""}

print(f"Resonance frequency: {f_res:.9f} GHz")
print(f"Energy splitting: {spin.E_All[1] - spin.E_All[0]:.9f} meV")

fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
for energy, label in zip(
    spin.E_All_inGHz, (r"$|g\rangle$", r"$|e\rangle$"), strict=True
):
    ax.hlines(energy, 0.2, 0.8, linewidth=2)
    ax.text(0.85, energy, label, va="center")
ax.annotate(
    "",
    xy=(0.5, spin.E_All_inGHz[1]),
    xytext=(0.5, spin.E_All_inGHz[0]),
    arrowprops={"arrowstyle": "<->", "color": "gray"},
)
ax.set(xlim=(0, 1.1), ylim=(-1, f_res + 1), ylabel="Relative energy / h (GHz)")
ax.set_xticks([])
ax.set_title("Static spin-1/2 energy levels")
plt.show()

## 2. Rotating-wave approximation

A linearly oscillating transverse field contains co-rotating and counter-rotating components. In a frame rotating at $f_{\mathrm{RF}}$, the latter oscillates near $2f_{\mathrm{RF}}$ and is neglected when $f_R,|\delta|\ll f_{\mathrm{res}}$.

With $\delta=f_{\mathrm{RF}}-f_{\mathrm{res}}$ and a suitable RF phase, the module's generator is

$$
\frac{H_{\mathrm{rot}}}{\hbar}
=2\pi\left(\delta S_{z'}+f_R S_{x'}\right).
$$

Here $f_R=\texttt{RabiRate}$ is an ordinary frequency in GHz, not an angular frequency. For a linearly oscillating magnetic field with peak amplitude $B_1$, $f_R=g\mu_B B_1/(2h)$. In ESR-STM, it can instead represent an effective electrically driven coupling; see [Yang et al. (2019)](https://doi.org/10.1126/science.aay6779).

`RotFrameSecular` returns the static rotating-frame generator in rad/ns. `calc_RabiOscillation` adds the transverse drive; do not add it a second time.

In [ ]:
B1_equivalent = (
    2 * rabi_rate / (spin.gfactorvector[0] * spin.muB * spin.meVtoGHzConversion)
)
print(f"RabiRate: {rabi_rate * 1e3:.1f} MHz")
print(f"Equivalent magnetic B1 peak amplitude: {B1_equivalent * 1e3:.3f} mT")
print(f"Drive/resonance frequency ratio: {rabi_rate / f_res:.2e}")
print(f"On-resonance pi-pulse duration: {1 / (2 * rabi_rate):.1f} ns")

### Fitting the oscillation

We fit each simulated trace to

$$
\langle S_{z'}(t)\rangle=C+A\cos(2\pi f_{\mathrm{fit}}t+\phi).
$$

The fitted period is $1/f_{\mathrm{fit}}$. The positive cosine amplitude $A$ is half the peak-to-peak spin contrast; on resonance, the ideal values are $A=1/2$ and $C=0$.

The notebook-local fitter uses an FFT initial estimate, not the analytical Rabi formula. The existing `helperfunctions.fit_oscillation` also fits a decay time, which is not identifiable for these undamped traces. Fit uncertainties below describe numerical residuals, not experimental measurement errors.

In [ ]:
def fit_rabi_oscillation(time, signal):
    """
    Fit an undamped cosine with a constant offset.

    Parameters
    ----------
    time : array_like
        Increasing, evenly spaced pulse durations in ns, starting at zero.
        At least five samples covering several oscillations are assumed.
    signal : array_like
        Finite, nonconstant spin expectation values matching time.

    Returns
    -------
    time_fit : numpy.ndarray
        1000 evenly spaced pulse durations in ns.
    signal_fit : numpy.ndarray
        Fitted dimensionless spin expectation values.
    popt : numpy.ndarray
        Parameters [A, frequency, phase, offset]. A is the positive
        cosine amplitude, frequency is in GHz, phase is in radians, and
        offset is dimensionless.
    perr : numpy.ndarray
        Formal one-standard-deviation parameter uncertainties, in the
        corresponding units.

    Notes
    -----
    The frequency is restricted to the interval between zero and the
    Nyquist frequency. This model assumes no relaxation or dephasing.
    Failures from scipy.optimize.curve_fit are propagated.
    """

    def cosine_model(t, amplitude, frequency, phase, offset):
        return offset + amplitude * np.cos(2 * np.pi * frequency * t + phase)

    time = np.asarray(time, dtype=float)
    signal = np.asarray(signal, dtype=float)
    dt = np.mean(np.diff(time))

    offset_guess = np.mean(signal)
    amplitude_guess = np.ptp(signal) / 2
    fft_size = 8 * time.size
    frequencies = np.fft.rfftfreq(fft_size, d=dt)
    spectrum = np.fft.rfft(signal - offset_guess, n=fft_size)
    peak_index = np.argmax(np.abs(spectrum[1:])) + 1

    p0 = [
        amplitude_guess,
        frequencies[peak_index],
        np.angle(spectrum[peak_index]),
        offset_guess,
    ]
    bounds = ([0, 0, -np.inf, -np.inf], [np.inf, 0.5 / dt, np.inf, np.inf])
    popt, pcov = curve_fit(
        cosine_model, time, signal, p0=p0, bounds=bounds, maxfev=20000
    )
    perr = np.sqrt(np.diag(pcov))

    time_fit = np.linspace(time[0], time[-1], 1000)
    signal_fit = cosine_model(time_fit, *popt)
    return time_fit, signal_fit, popt, perr

## 3. Rabi oscillations on resonance

For the prepared ground state,

$$
\langle S_{z'}(t)\rangle=\tfrac12\cos(2\pi f_Rt).
$$

A $\pi$ pulse at $t_\pi=1/(2f_R)$ inverts the spin. We pass the tilted-frame readout operator explicitly; the module's default operators are in the laboratory frame.

In [ ]:
spin.f_RF = f_res
spin.RabiRate = rabi_rate

time_on, result_on = CC.calc_RabiOscillation(
    spin,
    pulsewidth_array,
    e_ops=[Sz_readout],
    solver_options=solver_options,
)
signal_on = np.asarray(result_on.expect[0])
time_fit, signal_fit, popt_on, perr_on = fit_rabi_oscillation(time_on, signal_on)
amplitude_on, frequency_on, phase_on, offset_on = popt_on

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(time_on[::8], signal_on[::8], ".", ms=3, label="Simulation")
ax.plot(time_fit, signal_fit, "-", linewidth=1.5, label="Cosine fit")
ax.axvline(1 / (2 * rabi_rate), color="gray", linestyle=":", label=r"$\pi$ pulse")
ax.set(
    xlabel="Pulse duration (ns)",
    ylabel=r"$\langle S_{z'}\rangle$",
    title=f"On resonance: RabiRate = {rabi_rate * 1e3:.0f} MHz",
    ylim=(-0.55, 0.55),
)
ax.legend()
plt.show()

print("Quantity                 Fitted       Ideal")
print(f"Frequency (MHz)       {frequency_on * 1e3:10.6f}  {rabi_rate * 1e3:10.6f}")
print(f"Period (ns)           {1 / frequency_on:10.6f}  {1 / rabi_rate:10.6f}")
print(f"Cosine amplitude      {amplitude_on:10.6f}  {0.5:10.6f}")
print(f"Offset                {offset_on:10.6f}  {0.0:10.6f}")

## 4. Changing the Rabi rate

Increasing `RabiRate` shortens the oscillation period and the $\pi$-pulse duration. For this ideal, resonant spin, the oscillation amplitude stays at $1/2$.

We reuse the same `spin` object, changing only the drive strength. No static-Hamiltonian rebuild is needed because the static field is unchanged.

In [ ]:
spin.f_RF = f_res
rabi_rates = np.array([5.0, 10.0, 20.0, 40.0]) * 1e-3  # GHz
rate_fit_parameters = []
fig, axes = plt.subplots(
    2, 2, sharex=True, sharey=True, figsize=(10, 6), constrained_layout=True
)

for ax, rate in zip(axes.flat, rabi_rates, strict=True):
    spin.RabiRate = float(rate)
    time_rate, result_rate = CC.calc_RabiOscillation(
        spin,
        pulsewidth_array,
        e_ops=[Sz_readout],
        solver_options=solver_options,
    )
    trace = np.asarray(result_rate.expect[0])
    fit_time, fit_trace, parameters, _ = fit_rabi_oscillation(time_rate, trace)
    rate_fit_parameters.append(parameters)

    ax.plot(time_rate[::8], trace[::8], ".", ms=2, label="Simulation")
    ax.plot(fit_time, fit_trace, "-", linewidth=1, label="Fit")
    ax.set_title(
        f"RabiRate = {rate * 1e3:.0f} MHz; fitted T = {1 / parameters[1]:.1f} ns"
    )
    ax.set_ylim(-0.55, 0.55)

axes[0, 0].legend(fontsize=8)
fig.supxlabel("Pulse duration (ns)")
fig.supylabel(r"$\langle S_{z'}\rangle$")
plt.show()

rate_fit_parameters = np.asarray(rate_fit_parameters)
print("Input (MHz)   Fitted (MHz)   Fitted period (ns)")
for rate, parameters in zip(rabi_rates, rate_fit_parameters, strict=True):
    print(
        f"{rate * 1e3:10.2f}   {parameters[1] * 1e3:12.6f}   {1 / parameters[1]:18.6f}"
    )

# Restore the baseline before the detuning experiment.
spin.RabiRate = rabi_rate
spin.f_RF = f_res
CC.RotFrameSecular(spin, spin.f_RF);

## 5. Detuning map

Away from resonance, the generalized Rabi frequency is

$$
f_{\mathrm{osc}}=\sqrt{f_R^2+\delta^2}.
$$

For the initial ground state, the ideal spin signal is

$$
\langle S_{z'}(t)\rangle=
\frac{\delta^2}{2(f_R^2+\delta^2)}
+\frac{f_R^2}{2(f_R^2+\delta^2)}
\cos\!\left(2\pi\sqrt{f_R^2+\delta^2}\,t\right).
$$

Thus detuning speeds up the oscillation but reduces its contrast. We request serial execution with `n_jobs=1` to keep the example lightweight and reproducible; the returned map has frequency rows and pulse-duration columns.

In [ ]:
spin.RabiRate = rabi_rate
spin.f_RF = f_res
detuning_array = np.linspace(-50.0, 50.0, 81) * 1e-3  # GHz
frequency_array = f_res + detuning_array

time_map, frequencies_map, signal_map = CC.calc_RabiDetuningMap(
    spin,
    pulsewidth_array,
    frequency_array,
    e_ops=Sz_readout,
    solver_options=solver_options,
    n_jobs=1,
)
detuning_map = frequencies_map - f_res  # GHz

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
image = ax.pcolormesh(
    time_map,
    detuning_map * 1e3,
    signal_map,
    shading="auto",
    cmap="RdBu_r",
    vmin=-0.5,
    vmax=0.5,
)
ax.set(
    xlabel="Pulse duration (ns)",
    ylabel=r"Detuning $\delta=f_{\mathrm{RF}}-f_{\mathrm{res}}$ (MHz)",
    title=f"Rabi detuning map: RabiRate = {rabi_rate * 1e3:.0f} MHz",
)
fig.colorbar(image, ax=ax, label=r"$\langle S_{z'}\rangle$")
plt.show()

# Serial map evaluation leaves H_RF at the last frequency; restore it.
spin.f_RF = f_res
CC.RotFrameSecular(spin, spin.f_RF);

### Fitting the map traces

Each row is fitted independently using the same cosine model, including a free offset and phase. Representative traces show the quality of the fits before we extract period and amplitude.

In [ ]:
map_fit_parameters = []
map_fit_errors = []
fit_residual_rms = []

for trace in signal_map:
    fit_time, fit_trace, parameters, errors = fit_rabi_oscillation(time_map, trace)
    amplitude, frequency, phase, offset = parameters
    predicted = offset + amplitude * np.cos(2 * np.pi * frequency * time_map + phase)
    map_fit_parameters.append(parameters)
    map_fit_errors.append(errors)
    fit_residual_rms.append(np.sqrt(np.mean((trace - predicted) ** 2)))

map_fit_parameters = np.asarray(map_fit_parameters)
map_fit_errors = np.asarray(map_fit_errors)
fit_residual_rms = np.asarray(fit_residual_rms)

fig, axes = plt.subplots(1, 3, sharey=True, figsize=(12, 3.5), constrained_layout=True)
for ax, detuning_mhz in zip(axes, (-40.0, 0.0, 20.0), strict=True):
    index = np.argmin(np.abs(detuning_map * 1e3 - detuning_mhz))
    fit_time, fit_trace, _, _ = fit_rabi_oscillation(time_map, signal_map[index])
    ax.plot(time_map[::8], signal_map[index, ::8], ".", ms=2, label="Simulation")
    ax.plot(fit_time, fit_trace, "-", linewidth=1, label="Fit")
    ax.set(
        xlabel="Pulse duration (ns)",
        title=f"Detuning = {detuning_map[index] * 1e3:.0f} MHz",
        ylim=(-0.55, 0.55),
    )
axes[0].set_ylabel(r"$\langle S_{z'}\rangle$")
axes[0].legend(fontsize=8)
plt.show()

print(f"Largest RMS fit residual: {np.max(fit_residual_rms):.2e}")

## 6. Period and amplitude from the fits

The plotted points below come from the fit parameters, not from sampled extrema or zero crossings:

$$
T_{\mathrm{fit}}=\frac{1}{|f_{\mathrm{fit}}|},
\qquad
A_{\mathrm{fit}}=|A|.
$$

The analytical comparison is $T=1/\sqrt{f_R^2+\delta^2}$ and $A=f_R^2/[2(f_R^2+\delta^2)]$. At resonance, a 10 MHz drive gives a 100 ns period and a cosine amplitude of $1/2$. The peak-to-peak contrast is $2A$; for the prepared ground state, it also equals the maximum excited-state probability.

In [ ]:
fitted_period = 1 / np.abs(map_fit_parameters[:, 1])  # ns
fitted_amplitude = np.abs(map_fit_parameters[:, 0])
period_uncertainty = map_fit_errors[:, 1] / map_fit_parameters[:, 1] ** 2
amplitude_uncertainty = map_fit_errors[:, 0]

generalized_frequency = np.hypot(rabi_rate, detuning_map)
expected_period = 1 / generalized_frequency
expected_amplitude = 0.5 * rabi_rate**2 / generalized_frequency**2
expected_offset = 0.5 * detuning_map**2 / generalized_frequency**2

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].plot(detuning_map * 1e3, expected_period, "k-", label="Analytical RWA")
axes[0].errorbar(
    detuning_map * 1e3,
    fitted_period,
    yerr=period_uncertainty,
    fmt="o",
    ms=3,
    label="From cosine fits",
)
axes[0].set(
    xlabel="Detuning (MHz)",
    ylabel="Oscillation period (ns)",
    title="Fitted oscillation period",
)
axes[1].plot(detuning_map * 1e3, expected_amplitude, "k-", label="Analytical RWA")
axes[1].errorbar(
    detuning_map * 1e3,
    fitted_amplitude,
    yerr=amplitude_uncertainty,
    fmt="o",
    ms=3,
    label="From cosine fits",
)
axes[1].set(
    xlabel="Detuning (MHz)",
    ylabel=r"Cosine amplitude $A$",
    title="Fitted oscillation amplitude",
)
for ax in axes:
    ax.legend(fontsize=8)
plt.show()

### Numerical checks and scope

We compare the simulated map and the fitted quantities with the independent two-level RWA expressions. These checks provide a starting point for subsequent tests of `CoherentControl.py`.

This validates the ideal rotating-frame calculation, not the RWA against a full laboratory-frame simulation. Adding collapse operators or thermal preparation changes the signal and requires a different analytical comparison and, generally, a damped fit.

In [ ]:
expected_map = expected_offset[:, None] + expected_amplitude[:, None] * np.cos(
    2 * np.pi * generalized_frequency[:, None] * time_map[None, :]
)

np.testing.assert_allclose(signal_map, expected_map, rtol=0, atol=1e-8)
np.testing.assert_allclose(frequency_on, rabi_rate, rtol=1e-4)
np.testing.assert_allclose(amplitude_on, 0.5, atol=1e-6)
np.testing.assert_allclose(rate_fit_parameters[:, 1], rabi_rates, rtol=1e-4)
np.testing.assert_allclose(fitted_period, expected_period, rtol=1e-4)
np.testing.assert_allclose(fitted_amplitude, expected_amplitude, rtol=1e-4, atol=1e-6)
np.testing.assert_allclose(map_fit_parameters[:, 3], expected_offset, atol=1e-6)

print(f"Maximum map error: {np.max(np.abs(signal_map - expected_map)):.2e}")
print(
    "Maximum relative fitted-period error: "
    f"{np.max(np.abs(fitted_period / expected_period - 1)):.2e}"
)
print(
    "Maximum fitted-amplitude error: "
    f"{np.max(np.abs(fitted_amplitude - expected_amplitude)):.2e}"
)
print("All ideal spin-1/2 RWA checks passed.")

# Leave the same SpinSys instance at the baseline drive settings.
spin.RabiRate = rabi_rate
spin.f_RF = f_res
CC.RotFrameSecular(spin, spin.f_RF);

## Background literature

- I. I. Rabi, *Space Quantization in a Gyrating Magnetic Field*, Physical Review **51**, 652–654 (1937): the foundational rotating-field spin problem. [DOI](https://doi.org/10.1103/PhysRev.51.652)
- K. Yang et al., *Coherent spin manipulation of individual atoms on a surface*, Science **366**, 509–512 (2019): Rabi oscillations and pulsed coherent control in ESR-STM. [DOI](https://doi.org/10.1126/science.aay6779)

The parameters here are illustrative; this notebook is not a reproduction of the measured Yang et al. traces.